# 05 - Insights e recomendações

In [ ]:
from pathlib import Path
import sys
import warnings

import joblib
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from olist.config import INTERIM_DATA_DIR, MODELS_DIR, PROCESSED_DATA_DIR, REPORTS_DIR, ensure_project_dirs

ensure_project_dirs()

In [ ]:
order_path_parquet = INTERIM_DATA_DIR / 'order_level_dataset.parquet'
order_path_csv = INTERIM_DATA_DIR / 'order_level_dataset.csv'
modeling_path_parquet = PROCESSED_DATA_DIR / 'modeling_dataset.parquet'
modeling_path_csv = PROCESSED_DATA_DIR / 'modeling_dataset.csv'

order_level = pd.read_parquet(order_path_parquet) if order_path_parquet.exists() else pd.read_csv(order_path_csv)
modeling_df = pd.read_parquet(modeling_path_parquet) if modeling_path_parquet.exists() else pd.read_csv(modeling_path_csv)

print(order_level.shape, modeling_df.shape)

## KPIs principais


In [ ]:
delivered = order_level[order_level['order_status'].eq('delivered')].copy()

kpis = pd.DataFrame({
    'metric': [
        'orders_total', 'orders_delivered', 'customers_unique', 'products_value_delivered',
        'avg_review_score', 'low_review_rate', 'late_rate', 'avg_delivery_days'
    ],
    'value': [
        order_level['order_id'].nunique(),
        delivered['order_id'].nunique(),
        order_level['customer_unique_id'].nunique(),
        delivered['products_value'].sum(),
        delivered['review_score'].mean(),
        delivered['is_low_review'].mean(),
        delivered['is_late'].mean(),
        delivered['delivery_time_days'].mean(),
    ],
})

display(kpis)
kpis.to_csv(REPORTS_DIR / 'executive_kpis.csv', index=False)

In [ ]:
category_risk_path = REPORTS_DIR / 'category_risk_top15.csv'
state_risk_path = REPORTS_DIR / 'state_risk.csv'
seller_risk_path = REPORTS_DIR / 'seller_risk_top20.csv'

if category_risk_path.exists():
    display(pd.read_csv(category_risk_path).head(10))
if state_risk_path.exists():
    display(pd.read_csv(state_risk_path).head(10))
if seller_risk_path.exists():
    display(pd.read_csv(seller_risk_path).head(10))

## Usando o modelo para priorizar pedidos

In [ ]:
model_path = MODELS_DIR / 'low_review_risk_model.joblib'

if model_path.exists():
    artifact = joblib.load(model_path)
    feature_cols = artifact['feature_cols']
    proba = artifact['pipeline'].predict_proba(modeling_df[feature_cols])[:, 1]
    scored = modeling_df[['order_id', 'customer_state', 'dominant_category', 'review_score', 'is_low_review']].copy()
    scored['low_review_risk_score'] = proba
    scored['risk_decile'] = pd.qcut(scored['low_review_risk_score'].rank(method='first'), 10, labels=False) + 1
    top_risk = scored.sort_values('low_review_risk_score', ascending=False).head(100)
    display(top_risk.head(20))
    top_risk.to_csv(REPORTS_DIR / 'top_100_low_review_risk_orders.csv', index=False)
else:
    print('Modelo ainda nao encontrado. Execute o notebook 04 antes desta etapa.')

## Matriz de recomendações

In [ ]:
recommendations = pd.DataFrame([
    {
        'area': 'Logí­stica',
        'evidence_to_check': 'Atraso associado a maior taxa de review baixo',
        'recommended_action': 'Criar fila de monitoramento para pedidos com risco de atraso e comunicação proativa ao cliente.',
        'validation': 'Piloto em estados/categorias de maior risco com comparação antes/depois.',
    },
    {
        'area': 'Seller success',
        'evidence_to_check': 'Vendedores com volume relevante e baixa avaliação média',
        'recommended_action': 'Priorizar sellers para revisão de SLA, embalagem, estoque e despacho.',
        'validation': 'Acompanhar evolução de atraso e review por seller apos intervenção.',
    },
    {
        'area': 'Pricing e frete',
        'evidence_to_check': 'Frete relativo alto associado a pior experiência',
        'recommended_action': 'Testar regras de subsí­dio, comunicação ou agrupamento de frete em faixas críticas.',
        'validation': 'Teste controlado medindo conversão, margem e review.',
    },
    {
        'area': 'Produto analytics',
        'evidence_to_check': 'Modelo ranqueia pedidos de risco melhor que baseline',
        'recommended_action': 'Transformar score em dashboard operacional com capacidade diária definida.',
        'validation': 'Medir precisao, recall e ganho operacional mensalmente.',
    },
])

display(recommendations)
recommendations.to_csv(REPORTS_DIR / 'recommendations_matrix.csv', index=False)

In [ ]:
kpi_lookup = dict(zip(kpis['metric'], kpis['value']))

summary_md = f"""# Executive summary - Olist ecommerce analytics

## Contexto

Analise do dataset público da Olist para entender satisfação, atraso logístico e oportunidades de priorização operacional.

## KPIs

- Pedidos totais: {kpi_lookup['orders_total']:,.0f}
- Pedidos entregues: {kpi_lookup['orders_delivered']:,.0f}
- Clientes únicos: {kpi_lookup['customers_unique']:,.0f}
- Valor de produtos entregues: R$ {kpi_lookup['products_value_delivered']:,.2f}
- Review médio: {kpi_lookup['avg_review_score']:.2f}
- Taxa de review baixo: {kpi_lookup['low_review_rate']:.2%}
- Taxa de atraso: {kpi_lookup['late_rate']:.2%}
- Prazo médio de entrega: {kpi_lookup['avg_delivery_days']:.1f} dias

## Recomendações iniciais

1. Priorizar gestão de atraso em segmentos com maior taxa de review baixo.
2. Investigar categorias, estados e sellers com volume suficiente e desempenho abaixo da média.
3. Usar o modelo de risco como fila de priorização, não como decisão automatica.
4. Validar qualquer ação com piloto operacional ou teste controlado.
"""

summary_path = REPORTS_DIR / 'executive_summary.md'
summary_path.write_text(summary_md, encoding='utf-8')
display(Markdown(summary_md))
print(f'Resumo salvo em: {summary_path}')